# Getting Started with Fine-Tuning Moshi 7B

This notebook shows you a simple example of how to LoRA finetune Moshi 7B. You can run this notebook in Google Colab using a A100 GPU.

<a target="_blank" href="https://colab.research.google.com/github//kyutai-labs/moshi-finetune/blob/main/tutorials/moshi_finetune.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Check out `moshi-finetune` Github repo to learn more: https://github.com/kyutai-labs/moshi-finetune/


> **Note before you start:** this is Kyutai's own official notebook, unmodified — I've only added explanation cells like this one between their original cells (the code cells below are exactly theirs). Their notebook metadata targets an **A100 GPU** (Colab Pro/Pro+, not the free T4 tier) — Moshi is a 7B-parameter model, and that's the GPU size they themselves tested this on. Everything below will still teach you the real pipeline even if you're on a smaller GPU; just know that's the gap you're working around if you hit memory errors.


## Installation

Clone the `moshi-finetune` repo:


**What this does:** downloads Kyutai's `moshi-finetune` codebase into `/content/moshi-finetune` on the Colab machine. This is the actual training code (config parser, LoRA logic, checkpoint saving) — nothing here is Moshi's model weights yet, those come later and are much bigger.


In [ ]:
%cd /content/
!git clone https://github.com/kyutai-labs/moshi-finetune.git

Install all required dependencies:


**What this does:** installs the repo and its Python dependencies in "editable" mode, so `import finetune` and `-m train` work from anywhere.

**If you get a `ResolutionImpossible` / dependency-conflict error here:** this is a known pain point with pip on this repo — the maintainers' own README recommends using `uv` instead, since it resolves the same `pyproject.toml` far more reliably. If the cell below fails, replace it with:
```
!pip install -q uv
!uv pip install --system -q -e /content/moshi-finetune
```
and re-run. That's the documented fix, not a workaround.


In [ ]:
%pip install -e /content/moshi-finetune

## Prepare dataset


**What this does:** downloads Kyutai's own `DailyTalkContiguous` dataset — real stereo conversation audio already in the exact format the training code expects (left channel = Moshi, right channel = the other speaker, plus a `.json` transcript per file). It's about **14GB**, so this cell can take a while depending on Colab's connection that day.

This is the point where you'd swap in your own small dataset instead, once you've seen this pipeline work with the sample data — same stereo-wav + transcript-json format, referenced from a `.jsonl` index.


In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

Path("/content/data/daily-talk-contiguous").mkdir(parents=True, exist_ok=True)

# Download the dataset
local_dir = snapshot_download(
    "kyutai/DailyTalkContiguous",
    repo_type="dataset",
    local_dir="/content/data/daily-talk-contiguous",
)

## Start training


**What this does:** pins training to GPU 0 with a stable PCI ordering — mostly relevant on multi-GPU machines, harmless (and a bit of a no-op) on a single-GPU Colab instance, but it's what the official notebook sets.


In [ ]:
# these info is needed for training
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

**What this does:** writes the training config to `/content/example.yaml`. This is the part worth actually reading, since these are the knobs that decide whether it fits in your GPU's memory and how long it takes:

| Setting | What it controls | Effect on memory/time |
|---|---|---|
| `lora.rank` | Size of the trainable LoRA adapters (128 here) | Higher = more capacity to adapt, more memory |
| `duration_sec` | Max length (seconds) of each training audio clip | Higher = more context learned per step, more memory |
| `batch_size` | Examples processed per step | Higher = faster convergence, more memory |
| `max_steps` | Total training steps | Higher = longer run, more chance to actually learn something |
| `gradient_checkpointing` | Trades compute for memory | `true` = slower steps, notably less memory |
| `save_adapters` | Save only the small LoRA weights vs. a full merged model | `true` = much smaller, faster checkpoint |

The defaults here (`duration_sec: 100`, `batch_size: 1`, `max_steps: 300`) are Kyutai's own "getting started" values for an A100. On a smaller GPU, `duration_sec` is the first thing to lower if you hit an out-of-memory error, followed by `lora.rank`.


In [ ]:
# define training configuration
# for your own use cases, you might want to change the data paths, model path, run_dir, and other hyperparameters
import yaml

config = """
# data
data:
  train_data: '/content/data/daily-talk-contiguous/dailytalk.jsonl' # Fill
  eval_data: '' # Optionally Fill
  shuffle: true

# model
moshi_paths:
  hf_repo_id: "kyutai/moshiko-pytorch-bf16"


full_finetuning: false # Activate lora.enable if partial finetuning
lora:
  enable: true
  rank: 128
  scaling: 2.
  ft_embed: false

# training hyperparameters
first_codebook_weight_multiplier: 100.
text_padding_weight: .5


# tokens per training steps = batch_size x num_GPUs x duration_sec
# we recommend a sequence duration of 300 seconds
# If you run into memory error, you can try reduce the sequence length
duration_sec: 100
batch_size: 1
max_steps: 300

gradient_checkpointing: true # Activate checkpointing of layers

# optim
optim:
  lr: 2.e-6
  weight_decay: 0.1
  pct_start: 0.05

# other
seed: 0
log_freq: 10
eval_freq: 1
do_eval: False
ckpt_freq: 10

save_adapters: True

run_dir: "/content/test"  # Fill
"""

# save the same file locally into the example.yaml file
with open("/content/example.yaml", "w") as file:
    yaml.dump(yaml.safe_load(config), file)

In [ ]:
# make sure the run_dir has not been created before
# only run this when you ran torchrun previously and created the /content/test_ultra file
# ! rm -r /content/test

**What this does:** nothing by default (it's commented out) — it's just a reminder that `run_dir` (`/content/test`) must not already exist when training starts. If you re-run training after an earlier attempt, uncomment this line first to clear the old run directory, or change `run_dir` in the config cell above to a fresh path.


In [ ]:
# start training

!cd /content/moshi-finetune && torchrun --nproc-per-node 1 -m train /content/example.yaml

**What this does:** actually runs training. Expect it to:
1. First download the base `kyutai/moshiko-pytorch-bf16` model weights (~14GB, once) if not already cached.
2. Print a log line roughly every `log_freq` steps (10, per the config above) showing the current step and loss.
3. Save a checkpoint under `/content/test/checkpoints/` every `ckpt_freq` steps (10, per the config above).
4. Print something like `Closed everything!` when the full `max_steps` run finishes.

This is the cell most likely to hit an out-of-memory error on anything smaller than an A100 — if it does, go back to the config cell, lower `duration_sec` (try 20–30) and/or `lora.rank` (try 8–16), re-run that cell, then re-run this one.


## Inference

Once the model has been trained, inference can be run on the colab GPU too, and gradio can be used to tunnel the audio data from a local client to the notebook.

More details on how to set this up can be found in the [moshi readme](https://github.com/kyutai-labs/moshi?tab=readme-ov-file#python-pytorch).


**What's next:** the cells below spin up Moshi's actual voice interface — using your fine-tuned LoRA weights — and expose it through a public Gradio tunnel link so you can literally talk to it from your browser, mic and all. This is the step that turns a saved checkpoint into something you can actually hear.


In [ ]:
!pip install gradio

**What this does:** installs Gradio, which the next cell uses to tunnel real-time audio between your browser's mic/speakers and the model running on the Colab GPU.


In [ ]:
!python -m moshi.server --gradio-tunnel --lora-weight=/content/test/checkpoints/checkpoint_000300/consolidated/lora.safetensors --config-path=/content/test/checkpoints/checkpoint_000300/consolidated/config.json